# Homework 01B — A Softmax classifier from scratch

**Two sessions · one linear layer · manual gradients · four autograded TODOs**

You will make every important operation visible: pixels become scores, scores become probabilities, cross-entropy becomes a scalar loss, and derivatives become parameter updates. No automatic differentiation and no machine-learning classifier are allowed.

> Inspired by Stanford CS231n's [linear-classifier notes](https://cs231n.github.io/linear-classify/) and [assignment 1](https://cs231n.github.io/assignments2024/assignment1/).

## Learning goals

By the end, you should be able to trace the shapes through a linear classifier, explain why Softmax subtracts a row maximum, derive and check a gradient, distinguish learning rate from regularization strength, and explain why backpropagation becomes necessary for deeper models.

In [ ]:
import pickle
import tarfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

SEED = 42
CLASS_NAMES = np.array([
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
])
print('NumPy', np.__version__)

## 1. CIFAR-10 and an honest data split

The loader downloads the official CIFAR-10 Python archive once. We reserve 1,000 training images for validation and never use the official test set during model selection.

In [ ]:
def load_cifar10(cache_dir='data'):
    cache = Path(cache_dir)
    archive = cache / 'cifar-10-python.tar.gz'
    root = cache / 'cifar-10-batches-py'
    cache.mkdir(parents=True, exist_ok=True)
    if not root.exists():
        if not archive.exists():
            print('Downloading CIFAR-10 (about 163 MB)...')
            urllib.request.urlretrieve(
                'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', archive
            )
        with tarfile.open(archive, 'r:gz') as tar:
            tar.extractall(cache, filter='data')

    def read_batch(path):
        with open(path, 'rb') as handle:
            batch = pickle.load(handle, encoding='bytes')
        images = batch[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        return images, np.asarray(batch[b'labels'])

    parts = [read_batch(root / f'data_batch_{i}') for i in range(1, 6)]
    X_train = np.concatenate([p[0] for p in parts])
    y_train = np.concatenate([p[1] for p in parts])
    X_test, y_test = read_batch(root / 'test_batch')
    return X_train, y_train, X_test, y_test

X_all_images, y_all, X_test_images, y_test = load_cifar10()
rng = np.random.default_rng(SEED)
order = rng.permutation(len(X_all_images))
train_idx, val_idx = order[:49000], order[49000:]
X_train_images, y_train = X_all_images[train_idx], y_all[train_idx]
X_val_images, y_val = X_all_images[val_idx], y_all[val_idx]

def prepare(images):
    return images.reshape(len(images), -1).astype(np.float32) / 255.0

X_train, X_val, X_test = map(prepare, [X_train_images, X_val_images, X_test_images])
# Center using training statistics only.
pixel_mean = X_train.mean(axis=0, keepdims=True)
X_train -= pixel_mean
X_val -= pixel_mean
X_test -= pixel_mean
print(X_train.shape, X_val.shape, X_test.shape)

## TODO 1 — Forward: one matrix, ten scores

For a batch $X\in\mathbb{R}^{B\times D}$, weights $W\in\mathbb{R}^{D\times C}$, and bias $b\in\mathbb{R}^{C}$, implement

$$S=XW+b, \qquad S\in\mathbb{R}^{B\times C}.$$

Each row contains the ten class scores for one image. NumPy broadcasts `b` across the batch.

In [ ]:
def affine_scores(X, W, b):
    # TODO 1: compute all class scores for all instances.
    raise NotImplementedError('TODO 1')

In [ ]:
def check_todo_1(fn):
    X = np.array([[1., 2.], [-1., 3.]])
    W = np.array([[2., 0., -1.], [1., 4., 2.]])
    b = np.array([.5, -1., 2.])
    expected = np.array([[4.5, 7., 5.], [1.5, 11., 9.]])
    got = fn(X, W, b)
    assert got.shape == (2, 3)
    np.testing.assert_allclose(got, expected)
    print('✓ TODO 1 passed: affine scores and bias broadcasting')

check_todo_1(affine_scores)

## 2. The max trick: change the numbers, preserve the probabilities

Softmax is unchanged when the same constant is added to every score in one row:

$$\operatorname{softmax}(s+c)=\operatorname{softmax}(s).$$

Dividing numerator and denominator by $e^c$ cancels the shift. We exploit this by choosing $c=-\max_j s_j$, making the largest shifted score zero. First run the small safe example; then observe what happens to a naive exponential on large scores.

In [ ]:
def naive_softmax_1d(scores):
    exp_scores = np.exp(scores)
    return exp_scores / exp_scores.sum()

small = np.array([2., 1., -1.])
print('original:    ', naive_softmax_1d(small))
print('shifted +100:', naive_softmax_1d(small + 100))
print('same?', np.allclose(naive_softmax_1d(small), naive_softmax_1d(small + 100)))

large = np.array([1000., 999., 997.])
with np.errstate(over='ignore', invalid='ignore'):
    print('naive large scores:', naive_softmax_1d(large))
print('shifted scores:', large - large.max())

## TODO 2 — Stable Softmax per instance

Implement Softmax for a score matrix. Subtract each row's own maximum—not one maximum for the entire batch. Every output must be finite, positive, and each row must sum to one.

In [ ]:
def softmax(scores):
    # TODO 2: stable probabilities, independently for every row.
    raise NotImplementedError('TODO 2')

In [ ]:
def check_todo_2(fn):
    scores = np.array([[2., 1., -1.], [1000., 999., 997.]])
    probs = fn(scores)
    assert probs.shape == scores.shape
    assert np.all(np.isfinite(probs)) and np.all(probs > 0)
    np.testing.assert_allclose(probs.sum(axis=1), np.ones(2))
    np.testing.assert_allclose(probs[0], probs[1], atol=1e-12)
    np.testing.assert_allclose(fn(scores + np.array([[50.], [-300.]])), probs)
    print('✓ TODO 2 passed: stable, row-wise, shift-invariant Softmax')

check_todo_2(softmax)

### Prediction is already solved

The most probable class is also the class with the largest score. Prediction therefore needs neither Softmax nor a loop. We provide it so the new work stays focused on learning.

In [ ]:
def predict(X, W, b):
    return np.argmax(affine_scores(X, W, b), axis=1)

## TODO 3 — Loss and gradient by hand

For batch size $B$, average cross-entropy with L2 regularization is

$$L=-\frac{1}{B}\sum_i\log p_{i,y_i}+\lambda\lVert W\rVert_2^2.$$

The crucial derivative with respect to scores is surprisingly compact:

$$dS=\frac{P-\operatorname{onehot}(y)}{B},\quad dW=X^TdS+2\lambda W,\quad db=\sum_i dS_i.$$

Implement the loss and both gradients without looping over instances or classes. Return `(loss, dW, db)`. Clip the selected probability before `log` if necessary.

In [ ]:
def softmax_loss_and_gradients(X, y, W, b, reg=0.0):
    # TODO 3: forward loss followed by analytical gradients.
    raise NotImplementedError('TODO 3')

In [ ]:
def check_todo_3(fn):
    X = np.array([[1., -1.], [0.5, 2.], [-2., 1.]])
    y = np.array([0, 2, 1])
    W = np.array([[.1, -.2, .3], [-.1, .2, .05]])
    b = np.array([.01, -.02, .03])
    loss, dW, db = fn(X, y, W, b, reg=.1)
    assert np.ndim(loss) == 0 and np.isfinite(loss)
    assert dW.shape == W.shape and db.shape == b.shape
    np.testing.assert_allclose(loss, 0.8510310313, rtol=1e-6)
    np.testing.assert_allclose(dW, [[-.303600, .376984, -.033385], [.429953, .079702, -.479655]], rtol=2e-5, atol=2e-6)
    np.testing.assert_allclose(db, [-.043234, .050545, -.007311], rtol=2e-5, atol=2e-6)
    print('✓ TODO 3 public values passed; run the independent checker next')

check_todo_3(softmax_loss_and_gradients)

### Numerical gradient checker

A finite difference asks: if one parameter moves slightly up and down, does the measured loss slope agree with our formula? We check only selected entries because each numerical derivative requires two complete forward passes. Relative errors below approximately `1e-6` are strong evidence—not a proof—that the implementation is correct.

In [ ]:
def numerical_gradient_check(loss_fn, X, y, W, b, reg=.05, checks=12, h=1e-5, seed=0):
    loss, dW, db = loss_fn(X, y, W, b, reg)
    rng = np.random.default_rng(seed)
    errors = []
    for _ in range(checks):
        index = tuple(rng.integers(size) for size in W.shape)
        old = W[index]
        W[index] = old + h
        plus = loss_fn(X, y, W, b, reg)[0]
        W[index] = old - h
        minus = loss_fn(X, y, W, b, reg)[0]
        W[index] = old
        numeric = (plus - minus) / (2 * h)
        analytic = dW[index]
        relative = abs(numeric - analytic) / max(1e-8, abs(numeric) + abs(analytic))
        errors.append(relative)
    print(f'max relative error: {max(errors):.2e}')
    assert max(errors) < 1e-6, 'Gradient check failed. Inspect averaging, one-hot subtraction, and L2.'
    print('✓ numerical gradient check passed')

tiny_rng = np.random.default_rng(3)
tiny_X = tiny_rng.normal(size=(5, 7))
tiny_y = np.array([0, 2, 1, 2, 0])
tiny_W = tiny_rng.normal(scale=.01, size=(7, 3))
tiny_b = np.zeros(3)
numerical_gradient_check(softmax_loss_and_gradients, tiny_X, tiny_y, tiny_W, tiny_b)

**Reflection 1.** The numerical checker is simple and independent. Why do we not use it to train all 30,730 parameters? What extra difficulty appears when a model contains many layers?

## TODO 4 — One SGD update

Gradient descent moves parameters opposite the local uphill direction:

$$W\leftarrow W-\eta dW,\qquad b\leftarrow b-\eta db.$$

Implement an in-place update and return the updated pair. The learning rate $\eta$ controls step size; it is not part of the loss.

In [ ]:
def sgd_step(W, b, dW, db, learning_rate):
    # TODO 4: update both parameter arrays in place, then return W, b.
    raise NotImplementedError('TODO 4')

In [ ]:
def check_todo_4(fn):
    W = np.array([[1., -2.], [3., 4.]])
    b = np.array([.5, -.5])
    W_id, b_id = id(W), id(b)
    out_W, out_b = fn(W, b, np.ones_like(W), np.array([2., -1.]), .1)
    assert id(out_W) == W_id and id(out_b) == b_id, 'Update the supplied arrays in place.'
    np.testing.assert_allclose(W, [[.9, -2.1], [2.9, 3.9]])
    np.testing.assert_allclose(b, [.3, -.4])
    print('✓ TODO 4 passed: both parameters move opposite their gradients')

check_todo_4(sgd_step)

## 3. Training loop supplied

The loop below composes your functions. It samples mini-batches, computes loss and gradients, updates parameters, and records one loss per epoch. Read it carefully: future neural-network loops have the same skeleton.

In [ ]:
def train_softmax(X, y, *, learning_rate=1e-2, reg=1e-4, epochs=8,
                  batch_size=256, seed=42, verbose=False):
    rng = np.random.default_rng(seed)
    W = rng.normal(scale=1e-3, size=(X.shape[1], len(CLASS_NAMES)))
    b = np.zeros(len(CLASS_NAMES))
    history = []
    for epoch in range(epochs):
        order = rng.permutation(len(X))
        batch_losses = []
        for start in range(0, len(X), batch_size):
            idx = order[start:start + batch_size]
            loss, dW, db = softmax_loss_and_gradients(X[idx], y[idx], W, b, reg)
            sgd_step(W, b, dW, db, learning_rate)
            batch_losses.append(loss)
        history.append(float(np.mean(batch_losses)))
        if verbose:
            print(f'epoch {epoch + 1:02d} | loss {history[-1]:.4f}')
    return W, b, history

### Diagnostic: can the loss move in two epochs?

Random ten-class predictions begin near $-\log(0.1)\approx2.303$. Two epochs should already move the average loss downward. If it rises sharply or becomes `nan`, stop and diagnose before launching a grid search.

In [ ]:
probe_W, probe_b, probe_loss = train_softmax(
    X_train, y_train, learning_rate=5e-2, reg=1e-4, epochs=2, verbose=True
)
plt.plot([1, 2], probe_loss, marker='o')
plt.xticks([1, 2])
plt.xlabel('epoch')
plt.ylabel('mean mini-batch loss')
plt.title('Two-epoch learning diagnostic')
plt.grid(alpha=.2)
plt.show()

**Reflection 2.** Did the loss fall? Explain why a decreasing training loss is necessary evidence that the gradient works, but not evidence that the selected model generalizes.

## 4. Supplied grid search: learning rate × L2

We now compare optimization speed and model preference. Learning rate changes the path taken by SGD; L2 strength changes the objective itself. The validation set selects the pair. The test set remains untouched.

In [ ]:
def grid_search(X_train, y_train, X_val, y_val, learning_rates, regs, epochs=8):
    results = []
    best = None
    for lr in learning_rates:
        for reg in regs:
            W, b, losses = train_softmax(
                X_train, y_train, learning_rate=lr, reg=reg, epochs=epochs, seed=42
            )
            train_acc = float(np.mean(predict(X_train, W, b) == y_train))
            val_acc = float(np.mean(predict(X_val, W, b) == y_val))
            candidate = {
                'learning_rate': lr, 'reg': reg, 'W': W, 'b': b,
                'losses': losses, 'train_accuracy': train_acc, 'val_accuracy': val_acc
            }
            results.append(candidate)
            if best is None or val_acc > best['val_accuracy']:
                best = candidate
            print(f'lr={lr:.0e} reg={reg:.0e} | train={train_acc:.3f} val={val_acc:.3f}')
    return results, best

LEARNING_RATES = [1e-2, 5e-2, 1e-1]
REG_STRENGTHS = [0.0, 1e-4, 1e-3]
results, best = grid_search(
    X_train, y_train, X_val, y_val, LEARNING_RATES, REG_STRENGTHS
)
print('best:', {k: best[k] for k in ['learning_rate', 'reg', 'train_accuracy', 'val_accuracy']})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for candidate in results:
    label = f"lr={candidate['learning_rate']:.0e}, λ={candidate['reg']:.0e}"
    axes[0].plot(range(1, len(candidate['losses']) + 1), candidate['losses'], label=label)
axes[0].set(xlabel='epoch', ylabel='training loss', title='Optimization paths')
axes[0].grid(alpha=.2)
axes[0].legend(fontsize=7, ncol=2)

for reg in REG_STRENGTHS:
    subset = [r for r in results if r['reg'] == reg]
    axes[1].plot(LEARNING_RATES, [r['val_accuracy'] for r in subset], marker='o', label=f'λ={reg:.0e}')
axes[1].set_xscale('log')
axes[1].set(xlabel='learning rate', ylabel='validation accuracy', title='Selection evidence')
axes[1].grid(alpha=.2)
axes[1].legend()
plt.tight_layout()

**Reflection 3.** Identify one under-aggressive, one useful, and—if present—one unstable learning rate from the loss curves. How does L2 change the train/validation gap? Justify the selected candidate without using test accuracy.

## 5. Final test and learned class templates

Evaluate the selected candidate once. A linear classifier stores one weight vector per class; reshaping it into 32×32×3 produces a crude visual template of what raises that class's score.

In [ ]:
test_accuracy = np.mean(predict(X_test, best['W'], best['b']) == y_test)
print(f"train accuracy:      {best['train_accuracy']:.3%}")
print(f"validation accuracy: {best['val_accuracy']:.3%}")
print(f'test accuracy:       {test_accuracy:.3%}')

weights = best['W'].reshape(32, 32, 3, 10)
lo, hi = weights.min(), weights.max()
display_weights = (weights - lo) / (hi - lo + 1e-12)
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for label, ax in enumerate(axes.flat):
    ax.imshow(display_weights[:, :, :, label])
    ax.set_title(CLASS_NAMES[label])
    ax.axis('off')
plt.suptitle('One learned pixel template per class')
plt.tight_layout()

## Final diagnosis

1. Which templates contain recognizable colors or silhouettes? Which mix several appearances?
2. A single class has one weight template. Explain why this is limiting for objects seen from multiple viewpoints.
3. Trace the dimensions of `X`, `W`, `scores`, `probabilities`, `dS`, `dW`, and `db` for a batch of 256 images.
4. List the manual derivative steps required here. Predict how that burden changes after adding two hidden layers.
5. In one paragraph, explain how this exercise motivates backpropagation rather than merely introducing another classifier.

## Submission checklist

- [ ] Restart kernel and run all cells from top to bottom.
- [ ] Four green TODO checks and the numerical gradient check are visible.
- [ ] Two-epoch diagnostic, grid-search curves, and weight templates are visible.
- [ ] The test set was used only after hyperparameter selection.
- [ ] Reflections 1–3 and all final diagnosis questions are answered.